In [1]:
# check data quality
# check ranking constraints
# compare naive vs optimizer
# summarize final system outputs

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
INPUT_PATH = Path(
    "results/gta_promotion_rankings_v1.parquet"
)

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ECONOMIC_PATH = (
    OUTPUT_DIR /
    "gta_economic_rankings_v1.parquet"
)

OPTIMIZED_PATH = (
    OUTPUT_DIR /
    "gta_optimized_flyer_v1.parquet"
)

SUMMARY_PATH = (
    OUTPUT_DIR /
    "gta_optimized_flyer_summary_v1.csv"
)

In [4]:
df = pd.read_parquet(INPUT_PATH)

print("Shape:", df.shape)
print("Stores:", df["store_id"].nunique())
print("Columns:", df.columns.tolist())

display(df.head())

Shape: (105, 57)
Stores: 15
Columns: ['store_id', 'store_name', 'address', 'city', 'province', 'postal_code', 'full_address', 'latitude', 'longitude', 'universities', 'colleges', 'schools', 'offices', 'industrial', 'commercial', 'retail_landuse', 'rail_stations', 'store_name_demo', 'population_5km', 'population_density_5km', 'median_household_income_5km', 'average_household_size_5km', 'employment_rate_5km', 'n_census_tracts', 'product_id', 'product_name', 'category', 'current_price', 'regular_price', 'savings', 'promotion_type', 'product_url', 'page_number', 'snapshot_date', 'country', 'region', 'source', 'captured_at', 'discount_amount', 'discount_pct', 'is_promoted', 'promo_id', 'promo_price', 'context_id', 'discount_score', 'population_score', 'income_score', 'commercial_raw', 'commercial_score', 'weather_score', 'opportunity_score', 'store_rank', 'discount_contribution', 'population_contribution', 'commercial_contribution', 'income_contribution', 'weather_contribution']


,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude,universities,...,commercial_raw,commercial_score,weather_score,opportunity_score,store_rank,discount_contribution,population_contribution,commercial_contribution,income_contribution,weather_contribution
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,36.131981,4,0.000000,20.0,15.0,1.131981,0.0
1,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,50.293848,2,14.161868,20.0,15.0,1.131981,0.0
2,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,36.131981,5,0.000000,20.0,15.0,1.131981,0.0
3,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,43.717070,3,7.585090,20.0,15.0,1.131981,0.0
4,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,1631,1.0,0.0,81.131981,1,45.000000,20.0,15.0,1.131981,0.0


In [5]:
required = [
    "store_id",
    "product_name",
    "promo_price",
    "discount_pct",
    "opportunity_score"
]

missing = [
    col for col in required
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("Required columns OK")

Required columns OK


In [6]:
for col in [
    "promo_price",
    "regular_price",
    "discount_pct",
    "opportunity_score"
]:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

In [7]:
df = df[
    df["promo_price"].notna()
    & (df["promo_price"] > 0)
].copy()

print("Usable rows:", len(df))

Usable rows: 105


 # Business assumptions

In [8]:
ASSUMED_MARGIN_RATE = 0.25

MAX_INCREMENTAL_UNITS_PROXY = 10

MAX_FLYER_SLOTS = 20

# Economic proxy

In [9]:
df["estimated_unit_margin"] = (
    df["promo_price"]
    * ASSUMED_MARGIN_RATE
)

In [10]:
df["opportunity_normalized"] = (
    df["opportunity_score"]
    .clip(0, 100)
    / 100
)

In [9]:
print(
    "Opportunity outside 0–100:",
    (
        (rankings["opportunity_score"] < 0)
        |
        (rankings["opportunity_score"] > 100)
    ).sum()
)

Opportunity outside 0–100: 0


In [11]:
df["incremental_units_proxy"] = (
    df["opportunity_normalized"]
    * MAX_INCREMENTAL_UNITS_PROXY
)

In [12]:
df["profit_proxy"] = (
    df["incremental_units_proxy"]
    * df["estimated_unit_margin"]
)

# Normalize profit within each store

In [13]:
def normalize_group(series):

    min_v = series.min()
    max_v = series.max()

    if pd.isna(min_v) or pd.isna(max_v):
        return pd.Series(
            0.0,
            index=series.index
        )

    if max_v == min_v:
        return pd.Series(
            0.5,
            index=series.index
        )

    return (
        (series - min_v)
        / (max_v - min_v)
    )

In [14]:
df["profit_score"] = (
    df.groupby("store_id")[
        "profit_proxy"
    ]
    .transform(normalize_group)
)

pd.DataFrame(
    [metrics]
).to_csv(
    "results/final_validation_metrics.csv",
    index=False
)

# Discount penalty

In [15]:
df["discount_penalty"] = (
    df["discount_pct"]
    .fillna(0)
    .clip(0, 60)
    / 60
)

# Economic score

In [16]:
df["economic_score"] = (
    0.70 * df["profit_score"]
    +
    0.30 * df["opportunity_normalized"]
)

In [17]:
df["economic_score"] = (
    df["economic_score"]
    * (
        1
        - 0.15 * df["discount_penalty"]
    )
)

In [18]:
df["economic_score"] = (
    df["economic_score"]
    * 100
)

# Inspect

In [19]:
display(
    df[
        [
            "store_id",
            "product_name",
            "promo_price",
            "discount_pct",
            "opportunity_score",
            "estimated_unit_margin",
            "incremental_units_proxy",
            "profit_proxy",
            "economic_score"
        ]
    ]
    .sort_values(
        "economic_score",
        ascending=False
    )
    .head(20)
)

,store_id,product_name,promo_price,discount_pct,opportunity_score,estimated_unit_margin,incremental_units_proxy,profit_proxy,economic_score
0,3106,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,36.131981,24.250,3.613198,87.620053,80.839594
21,1150,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,29.066087,24.250,2.906609,70.485262,78.719826
14,1004,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,27.395502,24.250,2.739550,66.434093,78.218651
28,3105,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,20.530295,24.250,2.053029,49.785964,76.159088
98,3130,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,17.328743,24.250,1.732874,42.022202,75.198623
42,3055,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,17.280318,24.250,1.728032,41.904772,75.184096
56,1126,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,17.172037,24.250,1.717204,41.642191,75.151611
49,1061,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,15.866870,24.250,1.586687,38.477160,74.760061
77,1079,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,13.459188,24.250,1.345919,32.638530,74.037756
63,3054,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,11.425506,24.250,1.142551,27.706852,73.427652


# SAVE FULL ECONOMIC DATASET

In [20]:
df.to_parquet(
    ECONOMIC_PATH,
    index=False
)

print("Saved:", ECONOMIC_PATH)

print(
    "Has economic_score:",
    "economic_score" in df.columns
)

print(
    "Has profit_proxy:",
    "profit_proxy" in df.columns
)

Saved: results/gta_economic_rankings_v1.parquet
Has economic_score: True
Has profit_proxy: True


# Handle category constraint

In [21]:
if "category" not in df.columns:
    df["category"] = "Unknown"

print(
    df["category"]
    .value_counts(dropna=False)
)

category
Grocery    105
Name: count, dtype: int64


In [22]:
n_categories = df["category"].nunique(
    dropna=True
)

if n_categories <= 1:
    MAX_PRODUCTS_PER_CATEGORY = 20
else:
    MAX_PRODUCTS_PER_CATEGORY = 5

print(
    "Category limit:",
    MAX_PRODUCTS_PER_CATEGORY
)

Category limit: 20


# Optimizer

In [23]:
def optimize_store_flyer(
    store_df,
    max_slots=20,
    max_per_category=20
):

    store_df = (
        store_df
        .sort_values(
            "economic_score",
            ascending=False
        )
        .copy()
    )

    selected = []

    seen_products = set()
    category_counts = {}

    for _, row in store_df.iterrows():

        if (
            "product_id" in row.index
            and pd.notna(row["product_id"])
        ):
            product_key = str(
                row["product_id"]
            )
        else:
            product_key = str(
                row["product_name"]
            )

        if product_key in seen_products:
            continue

        category = row.get(
            "category",
            "Unknown"
        )

        if pd.isna(category):
            category = "Unknown"

        current_count = (
            category_counts.get(
                category,
                0
            )
        )

        if (
            current_count
            >= max_per_category
        ):
            continue

        selected.append(
            row.to_dict()
        )

        seen_products.add(
            product_key
        )

        category_counts[category] = (
            current_count + 1
        )

        if len(selected) >= max_slots:
            break

    result = pd.DataFrame(selected)

    if not result.empty:

        result["optimized_rank"] = (
            range(
                1,
                len(result) + 1
            )
        )

    return result

In [36]:
print("Rows in df:", len(df))

print(
    "Unique products total:",
    df["product_name"].nunique()
)

print("\nUnique products per store:")
display(
    df.groupby("store_id")["product_name"]
    .nunique()
    .to_frame("unique_products")
)

print("\nCategory counts:")
print(
    df["category"]
    .value_counts(dropna=False)
)

Rows in df: 105
Unique products total: 7

Unique products per store:


,unique_products
store_id,
1004,7
1061,7
1079,7
1126,7
1139,7
1150,7
1188,7
3031,7
3054,7



Category counts:
category
Grocery    105
Name: count, dtype: int64


In [37]:
print(
    "Rows with valid promo price:",
    (
        df["promo_price"].notna()
        & (df["promo_price"] > 0)
    ).sum()
)

Rows with valid promo price: 105


In [38]:
promos = pd.read_parquet(
    "data/external/walmart/promotions/processed/"
    "walmart_ca_flyer_snapshot.parquet"
)

print("Rows:", len(promos))
print("Unique products:", promos["product_name"].nunique())

display(
    promos[
        [
            "product_name",
            "current_price",
            "regular_price"
        ]
    ]
)

Rows: 7
Unique products: 7


,product_name,current_price,regular_price
0,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN
1,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",2.97,3.97
2,Quaker Chewy Chocolate Chips Granola Bars Vari...,10.97,13.44
3,Tim Hortons Original Blend Medium Roast Coffee...,19.86,25.47
4,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",5.98,9.97
5,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,1.93,NaN
6,"Red Bull Energy Drink, 250 ml (4 pack)",10.97,NaN


# Optimize every store

In [24]:
optimized_frames = []

for store_id, store_df in df.groupby(
    "store_id"
):

    result = optimize_store_flyer(
        store_df,
        max_slots=MAX_FLYER_SLOTS,
        max_per_category=MAX_PRODUCTS_PER_CATEGORY
    )

    if not result.empty:
        optimized_frames.append(result)

In [25]:
optimized = pd.concat(
    optimized_frames,
    ignore_index=True
)

print("Optimized shape:", optimized.shape)
print(
    "Stores:",
    optimized["store_id"].nunique()
)

Optimized shape: (105, 65)
Stores: 15


# Constraint validation

In [26]:
slots_per_store = (
    optimized
    .groupby("store_id")
    .size()
)

display(
    slots_per_store
    .to_frame("flyer_slots")
)

print(
    "Max flyer slots:",
    slots_per_store.max()
)

,flyer_slots
store_id,
1004,7
1061,7
1079,7
1126,7
1139,7
1150,7
1188,7
3031,7
3054,7


Max flyer slots: 7


In [27]:
category_check = (
    optimized
    .groupby(
        ["store_id", "category"]
    )
    .size()
    .reset_index(name="n")
)

print(
    "Max products/category:",
    category_check["n"].max()
)

Max products/category: 7


# Duplicate validation

In [28]:
print(
    "Duplicate store/product:",
    optimized.duplicated(
        subset=[
            "store_id",
            "product_name"
        ]
    ).sum()
)

Duplicate store/product: 0


In [29]:
example_store = (
    optimized["store_id"]
    .iloc[0]
)

example = (
    optimized[
        optimized["store_id"]
        == example_store
    ]
    .sort_values(
        "optimized_rank"
    )
)

display(
    example[
        [
            c for c in [
                "optimized_rank",
                "store_name",
                "product_name",
                "category",
                "promo_price",
                "discount_pct",
                "opportunity_score",
                "profit_proxy",
                "economic_score"
            ]
            if c in example.columns
        ]
    ]
)

,optimized_rank,store_name,product_name,category,promo_price,discount_pct,opportunity_score,profit_proxy,economic_score
0,1,Toronto Stockyards,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,27.395502,66.434093,78.218651
1,2,Toronto Stockyards,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,40.020060,72.395502,10.823128,28.738254
2,3,Toronto Stockyards,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,22.025913,34.980592,17.367864,26.216968
3,4,Toronto Stockyards,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,10.97,NaN,27.395502,7.513216,14.874799
4,5,Toronto Stockyards,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,18.377976,27.395502,7.513216,14.191377
5,6,Toronto Stockyards,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,25.188917,41.557370,3.085635,13.458918
6,7,Toronto Stockyards,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,Grocery,1.93,NaN,27.395502,1.321833,8.218651


# Store summary

In [30]:
store_summary = (
    optimized
    .groupby("store_id")
    .agg(
        flyer_products=(
            "optimized_rank",
            "count"
        ),

        total_profit_proxy=(
            "profit_proxy",
            "sum"
        ),

        avg_opportunity_score=(
            "opportunity_score",
            "mean"
        ),

        avg_economic_score=(
            "economic_score",
            "mean"
        ),

        avg_discount_pct=(
            "discount_pct",
            "mean"
        )
    )
    .reset_index()
)

display(store_summary)

,store_id,flyer_products,total_profit_proxy,avg_opportunity_score,avg_economic_score,avg_discount_pct
0,1004,7,114.058985,36.930782,26.273945,26.403217
1,1061,7,70.918843,25.402150,24.128474,26.403217
2,1079,7,61.909296,22.994467,23.936154,26.403217
3,1126,7,75.802779,26.707317,24.291653,26.403217
4,1139,7,51.328669,20.166935,24.001024,26.403217
5,1150,7,120.310315,38.601367,26.662664,26.403217
6,1188,7,48.354286,19.372070,24.110476,26.403217
7,3031,7,40.159366,17.182086,24.776522,26.403217
8,3054,7,54.299259,20.960785,23.938778,26.403217
9,3055,7,76.207967,26.815598,24.306640,26.403217


# Compare against naive largest-discount selection

In [31]:
naive_frames = []

for store_id, group in df.groupby(
    "store_id"
):

    naive = (
        group
        .sort_values(
            "discount_pct",
            ascending=False,
            na_position="last"
        )
        .drop_duplicates(
            subset=["product_name"]
        )
        .head(MAX_FLYER_SLOTS)
        .copy()
    )

    naive_frames.append(naive)

naive = pd.concat(
    naive_frames,
    ignore_index=True
)

In [32]:
comparison = pd.DataFrame({
    "method": [
        "Largest Discount",
        "Profit-Aware Optimizer"
    ],

    "total_profit_proxy": [
        naive["profit_proxy"].sum(),
        optimized["profit_proxy"].sum()
    ],

    "avg_opportunity_score": [
        naive["opportunity_score"].mean(),
        optimized["opportunity_score"].mean()
    ],

    "avg_economic_score": [
        naive["economic_score"].mean(),
        optimized["economic_score"].mean()
    ],

    "avg_discount_pct": [
        naive["discount_pct"].mean(),
        optimized["discount_pct"].mean()
    ]
})

display(comparison)

,method,total_profit_proxy,avg_opportunity_score,avg_economic_score,avg_discount_pct
0,Largest Discount,1124.450554,26.482996,24.806503,26.403217
1,Profit-Aware Optimizer,1124.450554,26.482996,24.806503,26.403217


In [33]:
naive_profit = (
    naive["profit_proxy"].sum()
)

optimized_profit = (
    optimized["profit_proxy"].sum()
)

if naive_profit != 0:

    profit_proxy_improvement_pct = (
        (
            optimized_profit
            - naive_profit
        )
        / naive_profit
        * 100
    )

else:
    profit_proxy_improvement_pct = np.nan

print(
    "Profit proxy improvement:",
    round(
        profit_proxy_improvement_pct,
        2
    ),
    "%"
)

Profit proxy improvement: 0.0 %


# Save final optimized flyer

In [39]:
optimized.to_parquet(
    OPTIMIZED_PATH,
    index=False
)

store_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Saved:", OPTIMIZED_PATH)
print("Saved:", SUMMARY_PATH)

Saved: results/gta_optimized_flyer_v1.parquet
Saved: results/gta_optimized_flyer_summary_v1.csv


# Final QA

In [40]:
print("=== FINAL 14 QA ===")

print(
    "Economic rows:",
    len(df)
)

print(
    "Optimized rows:",
    len(optimized)
)

print(
    "Stores:",
    optimized["store_id"].nunique()
)

print(
    "Max flyer slots:",
    slots_per_store.max()
)

print(
    "Duplicate store/product:",
    optimized.duplicated(
        subset=[
            "store_id",
            "product_name"
        ]
    ).sum()
)

print(
    "Economic score range:",
    round(
        df["economic_score"].min(),
        2
    ),
    "→",
    round(
        df["economic_score"].max(),
        2
    )
)

=== FINAL 14 QA ===
Economic rows: 105
Optimized rows: 105
Stores: 15
Max flyer slots: 7
Duplicate store/product: 0
Economic score range: 2.29 → 80.84
